# IN4080: obligatory assignment 3
 
Mandatory assignment 3 is about the practical use of Large Language Models (LLMs). More specifically, you will be tasked to implement a RAG (Retrieval-Augmented Generation) system able to answer factual questions based on a document database, more specifically Wiki pages extracted from an [online Star Wars encyclopedia](https://starwars.fandom.com). 

You are required to get at least 12/20 points to pass. 

- We assume that you have read and are familiar with IFI’s requirements and guidelines for mandatory assignments, see [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-mandatory.html) and [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-guidelines.html).
- This is an individual assignment. You should not deliver joint submissions. 
- You may redeliver in Devilry before the deadline (__Friday, October 23 at 23:59__).
- Only the last delivery will be read! If you deliver more than one file, put them into a zip-archive. You don't have to include in your delivery the data files already provided for this assignment. 
- Name your submission _your\_username\_in4080\_mandatory\_3_


<div class="alert alert-block alert-success"> <b>Practical setup:</b> The preferred way to complete this assignment is using the high-performance computing cluster <i>Fox</i>. See <a href="https://www.uio.no/studier/emner/matnat/ifi/IN4080/h25/computing-setup.html">here</a> for instructions on how to register and log in to Fox. 

Note: if you cannot work on Fox, you can do the assignment on your machine. In this case, make sure you have installed the following libraries: `pytorch`, `transformers`, `huggingface_hub`,  `bm25s`, `tf_keras`, `sentence-transformers` and either `nltk` or `spacy` (depending on how you prefer to do sentence splitting).
</div>

You should deliver a completed version of this Jupyter notebook, containing both your code and explanations about the steps you followed. We want to stress that simply submitting code is __not__ by itself sufficient to complete the assignment - we expect the notebook to also contain explanations of what you have implemented, along with motivations for the choices you made along the way. Preferably use whole sentences, and mathematical formulas if necessary. Explaining in your own words (using concepts we have covered through in the lectures) what you have implemented and reflecting on your solution is an important part of the learning process - take it seriously!

Regarding the use of LLMs (ChatGPT or similar): you are allowed to use them as 'sparring partner', for instance to clarify something you have not understood. However, you are __not__ allowed to use them to generate solutions (either in part or in full) to the assignment tasks. 


The answer

The conclusion is written in a separate Markdown cell below the output.

My fox username  is ec-zejingw


In [2]:
import torch
import transformers
import bm25s
import sentence_transformers
import nltk

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Everything imported successfully!")




PyTorch: 2.14.1+cu130
Transformers: 5.19.0
Sentence Transformers: 6.1.0
CUDA available: True
GPU: NVIDIA A40
Everything imported successfully!


Note

The device is GPU NVIDIA A40


CUDA is available

## Basic setup

We will start by building a chatbot that directly answers user questions using an instruction-tuned LLM, without relying on any database. We will use the instruction-tuned version of the [Gemma 4 language model](https://huggingface.co/google/gemma-4-E2B-it) from Google, which is available on HuggingFace. 

_Note: feel free to switch to another model (such as the Llama, Qwen, Deepseek or Mistral models) if you wish to experiment with them._



**Task 1** (4 points): Drawing inspiration from the code examples on the [Gemma webpage](https://huggingface.co/google/gemma-4-E2B-it), implement the `__init__` and `get_response` methods. If you run the code on Fox with a GPU (or on a personal machine with a GPU), make sure that your code actually runs on the GPU. Do not forget to use our `cache_dir` for the course, where we have already downloaded the models for you. You should use `cache_dir` when initializing/loading the model.

In [3]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM #Use the method from webpage "AutoProcessor"


class BasicResponseGenerator:

    def __init__(
        self,
        model_name="google/gemma-4-E2B-it",
        cache_dir="/fp/projects01/ec403/hf_models",
    ):
        """Loads the tokenizer and pretrained causal LM for the given model.
        If a GPU is available, the model should be loaded on the GPU"""

        #GPU or CPU
        #But I will use Fox
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

        #AutoProcessor
        self.processor = AutoProcessor.from_pretrained(model_name, cache_dir=cache_dir)
        
        self.model = AutoModelForMultimodalLM.from_pretrained(model_name, cache_dir=cache_dir, dtype="auto").to(self.device)

    #Generate the answer 
    def get_response(self, prompt: str, max_length: int = 50) -> str:
        """Given a prompt, generate a response (of a maximum max_length tokens) and return it.
        Only the response should be returned, not the text of the prompt itself
        """

        #Tokenize prompt
        messages = [{"role": "user", "content": prompt}]
        inputs = self.processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_dict=True, return_tensors="pt")

        #Move input to GPU or CPU
        inputs = inputs.to(self.device)

        #count input tokens 
        inputs_length = inputs["input_ids"].shape[1]#Get the number of input tokens

        #Generate without gradients

        with torch.no_grad():
            outputs = self.model.generate( **inputs, max_new_tokens=max_length )

        #Remove the prompt tokens because we only want the generated answer
        response_tokens = outputs[0, inputs_length:]

        #Decode token IDs into text 
        response = self.processor.decode(response_tokens, skip_special_tokens=True)
        return response.strip()


agent = BasicResponseGenerator()



print("Device:", agent.device)

answer = agent.get_response( "Who is Luke Skywalker?" )

print("Answer:", answer)




Loading weights: 100%|██████████| 1951/1951 [00:05<00:00, 348.22it/s]


Device: cuda
Answer: **Luke Skywalker is the protagonist of the original *Star Wars* film trilogy.** He is arguably one of the most iconic and important characters in science fiction cinema.

Here is a breakdown of who he is:

### Key Aspects of Luke Skywalker:


Conclusion


The model runs on CUDA (GPU) on Fox.

The model can generate an answer to the question "Who is Luke Skywalker?".

It identifies Luke Skywalker as the protagonist of the original Star Wars trilogy.

The AutoProcessor is used to prepare the input and convert the text into tokens. The model then generates new tokens based on the input.

After generation, the output tokens are decoded into text, and the original prompt is removed.

The maximum number of new tokens is set to 50.

This limits the length of the generated answer, and the response may be incomplete.

This shows that the model can run and generate a response.

The result shows that the model can successfully generate text,

but the answer length is limited by the generation settings.

_Note: An easy way to verify that the GPU is actually used is to run the command `nvidia-smi` while your code is running. There also exists alternative GPU monitoring tools, like [`gpustat`](https://pypi.org/project/gpustat/0.3.2/)._

You can then test your response generator with the following set of questions: 

In [4]:
questions = [
    "Who is Luke Skywalker?",
    "Where is the Niima Outpost in Star Wars?",
    "Have you heard of Nute Gunray? Who is he?",
    "What kind of planet is Kashyyyk, and who discovered it?",
    "Who are Condlurans, and can you give 2-3 names of known Condlurans?",
    "What can you tell me about the First Battle of Geonosis?",
    "What is the name of the settlement where Anakin Skywalker and his mother lived?",
    "Which planet did Darth Sidious represent as senator?",
]

for question in questions:
    print("Question:", question)
    print("Answer:", agent.get_response(question))
    print("-------")



Question: Who is Luke Skywalker?
Answer: **Luke Skywalker is the protagonist of the original *Star Wars* trilogy.**

Here is a detailed breakdown of who he is:

### Key Roles and Identity

* **Hero and Protagonist:** Luke is the central figure of the story, embarking
-------
Question: Where is the Niima Outpost in Star Wars?
Answer: I do not have specific, detailed information about a location called the "Niima Outpost" within the Star Wars universe in my current knowledge base.

It's possible that:

1. **It is from a very specific or niche source
-------
Question: Have you heard of Nute Gunray? Who is he?
Answer: I do not have specific, readily available information about an individual named "Nute Gunray" in my general knowledge base.

To help me understand who you are referring to, could you please provide some context? For example, is Nute Gun
-------
Question: What kind of planet is Kashyyyk, and who discovered it?
Answer: Kashyyyk is a **desert planet**.

It was discovered by **Lu

Conclusion

I tested the model with 8 different Star Wars questions.

The model can generate answers, but not all answers are correct.

For example, in questions 2, 3, and 5, the model failed to provide answers.

This shows that the model can generate text, but it is not always reliable.

If your implementation is correct, the model should give you a few correct answers, but also many responses for which the model is either unable to give a precise answer, or hallucinates a (wrong) answer. This is expected, as the model is relatively small (a few billion parameters) and is a generic model that is not particularly optimised to generate trivia about the Star Wars Franchise. We will now try to improve the model performance by coupling the LLM to a document database.

## Retrieval step

Retrieval-augmented generation operates on a simple idea: instead of directly generating a response based on the "parametric knowledge" of the LLM, we first search for relevant documents in a database (or on the web). We then include the most relevant documents to the prompt, and ask the LLM to answer the user question _based on this retrieved knowledge_. 

In this assignment, you will use a set of Wiki texts extracted from an [online Star Wars encyclopedia](https://starwars.fandom.com) as document database. The wiki texts are available as a JSON file [here](https://home.nr.no/~plison/data/starwars.json). The JSON is simply a dictionary mapping Wiki page titles to their content (in plain text).

### Sparse retrieval 

We can start by using the [BM25s](https://bm25s.github.io/) library, which implements a number of well-known search algorithms, which are all variants of the original [BM25 algorithm](https://en.wikipedia.org/wiki/Okapi_BM25) . Although BM25 is an old-fashioned search technique based on bag-of-words, it remains suprisingly effective, and is still widely used in modern NLP systems.

**Task 2** (4 points): Fill in the implementation for the `BM25Retriever` class using [BM25s](https://bm25s.github.io/) (see the library documentation for details). You should filter out stop words by adding `stopwords='en_plus'` to the arguments of the tokenizer. 

In [5]:
import bm25s
import json
from typing import List


class BM25Retriever:

    def __init__(self, filename="starwars.json"):
        """Using the json file provided as input, create a BM25s retriever
        containing all (indexed) documents."""

        with open(filename,"r",encoding ="utf-8") as f: data = json.load(f)
        
        print(type(data));print(str(data)[:500])
        
        self.corpus = list(data.values()) #Read the Wiki documents from the JSON file
        
        tokens = bm25s.tokenize(self.corpus,stopwords= "en_plus") #Tokenization

        self.retriever = bm25s.BM25(corpus=self.corpus); self.retriever.index(tokens)

       

    def search(self, query: str, k: int = 5) -> List[str]:
        """Use the BM25 retriever to find the k documents that are closest
        to the provided query"""

        query_tokens = bm25s.tokenize([query], stopwords="en_plus")

        results, scores = self.retriever.retrieve(query_tokens, k=k)

        return list(results[0])

In [6]:
import urllib.request; urllib.request.urlretrieve("https://home.nr.no/~plison/data/starwars.json", "starwars.json")

('starwars.json', <http.client.HTTPMessage at 0x14e78de84dd0>)

We can then test our retriever by checking whether the documents with highest BM25 scores are indeed the ones that are most relevant to the query:

In [7]:
retriever = BM25Retriever()
for question in questions:
    print("Question:", question)
    print("Retrieved documents:")
    for relevant_doc in retriever.search(question):
        print("- " + relevant_doc.replace("\n", " "))
    print("===========")

<class 'dict'>
{'Brianna': 'Brianna, also known as "The Last Handmaiden," was a half-Human, half-Echani hybrid fighter of Jedi heritage. The illegitimate daughter to the famous Echani General Yusanis and Jedi Master Arren Kae, both of whom fought in the Mandalorian Wars, she later served along with her five half-sisters as the Handmaiden Sisters under Jedi Master Atris in the period following the Jedi Civil War. During her service to Atris at the Jedi Academy on Telos IV, Brianna met Meetra Surik, also known a


Question: Who is Luke Skywalker?
Retrieved documents:


- Luke Skywalker's lightsaber was the first lightsaber constructed by Luke Skywalker and the second one he owned.
- The Luke Skywalker X-Wing Mech was a mech piloted by Luke Skywalker. It shared a design with his X-wing and included a large lightsaber.
- Holoeditor was a type of holographic technology used by Cronal to study and imitate Luke Skywalker in 5 ABY.Luke Skywalker and the Shadows of Mindor
- Luke Skywalker and the Jedi's Revenge was a holothriller produced prior to 6 ABY, which purported to depict the duel between Luke Skywalker and Darth Vader aboard the second Death Star. Skywalker took exception to its historical inaccuracies, specifically its depiction of him killing Vader to avenge Palpatine's death at Vader's hands, which he characterized as "sick". In actuality the holothriller was created by Blackhole, in order to establish that Luke Skywalker was the next legitimate heir to the Empire. Blackhole was secretly planning on becoming Luke Skywalker during the Battle of M

- The Niima Outpost Militia was a law enforcement agency stationed at Niima Outpost on the desert planet of Jakku. It was led by the Kyuzo Constable Zuvio and his two cousins&mdash;Drego and Streehn.
- Bay Three was a docking bay in Niima Outpost on the planet Jakku. When the scavenger Rey took BB-8 to Niima Outpost, she told the droid there was a trader in Bay Three named Horvins who may have been willing to give BB-8 a lift offworld.
- Niima Outpost was a junkyard settlement on Jakku, a desert planet in the Western Reaches of the galaxy. The outpost was named for and founded by Niima the Hutt after the Battle of Jakku to capitalize on the new scavenging opportunities the battle created on the planet. Niima Outpost was the only spaceport on the planet, although it was referred more as a landing field rather than a spaceport.Rey's Survival Guide Scavengers, like Rey, salvaged materials from the technology leftover from the Battle of Jakku. Salvage made up the backbone of the economy, b

- Nute Gunray's citadel, also referred to as Nute Gunray's redoubt, was a large fortress located on the western hemisphere of Cato Neimoidia. It was used as a stronghold and storehouse by Nute Gunray, Viceroy of the Trade Federation.
- Lora Besh claimed to be the secret lover of Trade Federation Viceroy Nute Gunray; her book on the alleged affair, Gunray On Top, was a bestseller in the months preceding the Clone Wars.
- The Viceroy's collar was an item worn by the Viceroy of the Trade Federation. Nute Gunray wore the collar.
- The sovereign beetle was a species from Neimoidia. The patterns on its shell were the basis of the ornamentation on Nute Gunray's mechno-chair.Cloak of Deception
- In 44 BBY, then-Senator Nute Gunray used a Trade Federation shuttle of a different class than the Sheathipede-class transport shuttle.
Question: What kind of planet is Kashyyyk, and who discovered it?
Retrieved documents:


- Mock Shyr was a kind of broad leafed plant native to Kashyyyk.
- The kolvissh was a kind of flowering plant native to Kashyyyk. Mallatobuck once used some for her bridal veil.
- A rare and unique spice variant was discovered during the Galactic Civil War. This kind of spice appeared organicQuest: "Man Down!" and medicinal in nature.
- Kashyyyk, also known as Planet Wookiee C to some humans in the Core Worlds, was a wroshyr tree-covered forest planet located in the southwestern quadrant of the galaxy and the homeworld of the Wookiee species. Four millennia before the Battle of Yavin, Kashyyyk was discovered by the Czerka Corporation, who enslaved the Wookiee population and renamed the planet G5-623, later to Edean. Using superior technology, the company managed to enslave the Wookiees until an uprising drove the oppressors away. During the Clone Wars, Kashyyyk was a member of the Galactic Republic, and endured enslavement under the Galactic Empire, the Republic's successor. Later, dur

- Condlurans were a sentient species that existed within the galaxy. The Condluran Briff worked as a smuggler on Jedha during the High Republic Era.The High Republic (2022) 2 During the Imperial Era, Freck was a Condluran male who worked as a transport driver for the Galactic Empire on the mining planet Mapuzo.
- A core name was a shortened name used by Chiss. Members of the Chiss species used their core names rather than their full names for at least two reasons. Among Chiss, core names were used in all but the most formal settings. Chiss also gave their core names to members of other species, as non-Chiss had difficulties pronouncing full Chiss names.
- Ration bars were a type of food ration known by various names, including nutrient bars, protein bars, and supply bars; they were also known as sticks instead of bars. They were eaten in multiple eras of galactic history.
- Figg & Associates Bank and Trust was a banking conglomerate. They were known to give loans to entrepreneurs, capt

- The First Battle of Geonosis, also referred to as the Battle of Geonosis or the Battle on Geonosis, was the first major battle fought in 22 BBY between the Confederacy of Independent Systems and the Galactic Republic on Geonosis, marking the beginning of the three-year Clone Wars. It would be the first major combat of the Grand Army of the Republic, as well as the first major battle the Jedi had fought in years. The battle also caused the death of the notorious bounty hunter Jango Fett and the discovery of Count Dooku's dark side allegiance.
- Ronto was a clone trooper copilot who fought in the First Battle of Geonosis during the Clone Wars. During his time on Geonosis, Ronto flew in a LAAT/i with Skifter.
- The Battle of Rendili was a battle of the Clone Wars, occurring thirty months after the First Battle of Geonosis, in 20 BBY.
- The First Battle of Kamino, also known as the Defense of Kamino, the Assault on Kamino, or simply as the Battle of Kamino, took place two months after th

- Mos Espa was a spaceport settlement located on the desert world of Tatooine. The settlement included a number of commercial and workspace settings, as well as entertainment establishments such as the Mos Espa Grand Arena. During the Invasion of Naboo, Mos Espa was also home to a number of slaves, including Anakin Skywalker and his mother, Shmi.
- Finn's mother lived with him on Coruscant. In 32 BBY, she helped Anakin Skywalker return to the Jedi Temple after he helped her son repair their malfunctioning nanny droid.
- The Skywalker home was where the slaves Anakin Skywalker and his mother Shmi Skywalker Lars lived in Mos Espa on the desert planet Tatooine. By the time the handmaiden Sabé traveled to Tatooine on behalf of Padmé Amidala to free Shmi from slavery, Shmi no longer lived in the home and the door had a newly cut symbol of a white sun on it.Queen's Shadow
- Kovit was a settlement, nestled within a steep mountain range on Trulalis where Jaalib Brandl and his mother once lived

- The massacre at the Gran Protectorate Embassy took place in 52 BBY. Senator Pax Teem of the Gran Protectorate was murdered in the embassy by Darth Plagueis' apprentice, Darth Sidious, thus removing an opponent to the Sith Lords.
- Dawk was a planet in the Outer Rim Territories that was orbited by a moon. During the Republic Era, between around 40 BBY and 32 BBY,The events of Darth Maul &ndash; Black, White & Red 3 take place during Darth Maul's time as the Sith apprentice of Darth Sidious, which Star Wars: Timelines dates to between around 40 BBY and 32 BBY. the Devaronian criminal Coir Cion operated on the moon of Dawk. The Sith Lord Darth Sidious dispatched his Sith apprentice, Darth Maul, to kill Cion and his associates after the Devaronian threatened to blackmail Senator Sheev Palpatine&mdash;Sidious's public persona&mdash;with information on his underworld dealings.
- The Useful Bureaucrats was a manifesto written by the Dark Lord of the Sith and Galactic Emperor Darth Sidious a

Conclusion

I used BM25 to retrieve documents from the Star Wars dataset.

I tested 8 questions. BM25 found relevant documents for most questions, such as Niima Outpost on Jakku.

However, some retrieved documents were not directly relevant to the question, especially for Luke Skywalker.

BM25 works well for word matching, but it does not always find the most relevant documents.

If your implementation is correct, the retrieved documents should for the most part relevant to the query. 

### Dense retrieval 

Many of those documents are, however, way too long to be included in the prompt for our Gemma model (especially if we wish to include 4-5 retrieved texts for each query!). Can we ensure that the length of each retrieved text stays within a reasonable length, such as one or two sentences? 

One strategy is to not return the full documents, but instead determine the most relevant _sentences_ within those documents. But how do we determine which sentence is most relevant? A sparse retriever using BM25 would not work well here, as it does not really account for the semantics of the query. Instead, what we can do is to:
- split the documents (retrieved through BM25) into sentences
- extract sentence embeddings for the query and for each sentence
- compute the cosine similarities between the query vector and each sentence vector
- and return the _k_ most similar sentences

In other words, our approach starts with a _sparse retrieval step_ at the level of full documents (which we already have implemented, using BM25S), and continues with a _dense retrieval step_ to determine the most relevant sentences among the sentences that are found in the retrieved documents.

**Task 3** (4 points): Re-implement the `search` method to segment into sentences each document retrieved with BM25, extract sentence embeddings for the query and sentences using the encoder model (see [here](https://sbert.net/examples/applications/semantic-search/README.html) for explanations and code examples), and then select the _k_ sentences with highest cosine similarities. Here, we are also loading the encoder model from our course `cache_dir`.

_Tips_: You can use `nltk.sent_tokenize` to segment your document in sentences. If you have problems with `nltk`, you can also use the sentence splitter from `spacy`, or anything else that works for you.

In [ ]:
import bm25s
import re, json
import sentence_transformers
import nltk

# nltk.download('punkt_tab')
from typing import List


class HybridRetriever(BM25Retriever):

    def __init__(
        self,
        filename="starwars.json",
        encoder_model="all-MiniLM-L6-v2",
        cache_dir="/fp/homes01/u01/ec-zejingw/hf_models",
    ):
        """Using the json file provided as input, create a BM25s retriever
        containing all (indexed) documents, and loads a sentence transformer model
        used to compute the embeddings for the query and sentences"""

        BM25Retriever.__init__(self, filename)
        self.encoder = sentence_transformers.SentenceTransformer(
            encoder_model, cache_folder=cache_dir
        )

    def search(self, query: str, k: int = 5) -> List[str]:
        """Use the BM25 retriever to find the documents that are closest
        to the provided query, and then the sentence transformer model to
        determine the most relevant sentences"""

        docs = BM25Retriever.search(self, query, k)

        sentences = []

        for doc in docs:

            try: parts = nltk.sent_tokenize(doc)

            except LookupError: parts = re.split(r"(?<=[.!?])\s+", doc)

            sentences.extend(s.strip() for s in parts if s.strip())

        if not sentences:
            return []
        query_embedding = self.encoder.encode(query, convert_to_tensor=True)

        sentence_embeddings = self.encoder.encode(sentences, convert_to_tensor=True, batch_size=32, show_progress_bar=False)

        similarities = sentence_transformers.util.cos_sim(query_embedding, sentence_embeddings)[0]

        top_indices = similarities.topk(min(k, len(sentences))).indices.tolist()

      #BM25 first retrieves relevant documents, and SentenceTransformer  then uses cosine similarity to select the most relevant sentences.
        return [sentences[i] for i in top_indices]

And we can test our hybrid (sparse followed by dense) retriever on the same questions as before:

In [9]:
retriever = HybridRetriever()
for question in questions:
    print("Question:", question)
    print("Retrieved documents:")
    for relevant_doc in retriever.search(question):
        print("- " + relevant_doc.replace("\n", " "))
    print("===========")




<class 'dict'>
{'Brianna': 'Brianna, also known as "The Last Handmaiden," was a half-Human, half-Echani hybrid fighter of Jedi heritage. The illegitimate daughter to the famous Echani General Yusanis and Jedi Master Arren Kae, both of whom fought in the Mandalorian Wars, she later served along with her five half-sisters as the Handmaiden Sisters under Jedi Master Atris in the period following the Jedi Civil War. During her service to Atris at the Jedi Academy on Telos IV, Brianna met Meetra Surik, also known a


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4993.80it/s]


Question: Who is Luke Skywalker?
Retrieved documents:


- Blackhole was secretly planning on becoming Luke Skywalker during the Battle of Mindor.
- Luke's Backpack was a blue colored satchel made on Dantooine out of gundark-skin that belonged to Luke Skywalker.
- Luke Skywalker's lightsaber was the first lightsaber constructed by Luke Skywalker and the second one he owned.
- The Luke Skywalker X-Wing Mech was a mech piloted by Luke Skywalker.
- In actuality the holothriller was created by Blackhole, in order to establish that Luke Skywalker was the next legitimate heir to the Empire.
Question: Where is the Niima Outpost in Star Wars?
Retrieved documents:


- Niima Outpost was a junkyard settlement on Jakku, a desert planet in the Western Reaches of the galaxy.
- Niima Outpost was the only spaceport on the planet, although it was referred more as a landing field rather than a spaceport.Rey's Survival Guide Scavengers, like Rey, salvaged materials from the technology leftover from the Battle of Jakku.
- The Niima Outpost Militia was a law enforcement agency stationed at Niima Outpost on the desert planet of Jakku.
- Bay Three was a docking bay in Niima Outpost on the planet Jakku.
- Niima resided in a temple on the desert planet of Jakku.
Question: Have you heard of Nute Gunray? Who is he?
Retrieved documents:


- Nute Gunray wore the collar.
- Lora Besh claimed to be the secret lover of Trade Federation Viceroy Nute Gunray; her book on the alleged affair, Gunray On Top, was a bestseller in the months preceding the Clone Wars.
- Nute Gunray's citadel, also referred to as Nute Gunray's redoubt, was a large fortress located on the western hemisphere of Cato Neimoidia.
- It was used as a stronghold and storehouse by Nute Gunray, Viceroy of the Trade Federation.
- In 44 BBY, then-Senator Nute Gunray used a Trade Federation shuttle of a different class than the Sheathipede-class transport shuttle.
Question: What kind of planet is Kashyyyk, and who discovered it?
Retrieved documents:


- Kashyyyk, also known as Planet Wookiee C to some humans in the Core Worlds, was a wroshyr tree-covered forest planet located in the southwestern quadrant of the galaxy and the homeworld of the Wookiee species.
- Four millennia before the Battle of Yavin, Kashyyyk was discovered by the Czerka Corporation, who enslaved the Wookiee population and renamed the planet G5-623, later to Edean.
- Kashyyyk wheat was a type of wheat grown on the planet of Kashyyyk.
- The kolvissh was a kind of flowering plant native to Kashyyyk.
- During the Clone Wars, Kashyyyk was a member of the Galactic Republic, and endured enslavement under the Galactic Empire, the Republic's successor.
Question: Who are Condlurans, and can you give 2-3 names of known Condlurans?
Retrieved documents:


- Condlurans were a sentient species that existed within the galaxy.
- The Condluran Briff worked as a smuggler on Jedha during the High Republic Era.The High Republic (2022) 2 During the Imperial Era, Freck was a Condluran male who worked as a transport driver for the Galactic Empire on the mining planet Mapuzo.
- Members of the Chiss species used their core names rather than their full names for at least two reasons.
- Chiss also gave their core names to members of other species, as non-Chiss had difficulties pronouncing full Chiss names.
- Among Chiss, core names were used in all but the most formal settings.
Question: What can you tell me about the First Battle of Geonosis?
Retrieved documents:


- The First Battle of Geonosis, also referred to as the Battle of Geonosis or the Battle on Geonosis, was the first major battle fought in 22 BBY between the Confederacy of Independent Systems and the Galactic Republic on Geonosis, marking the beginning of the three-year Clone Wars.
- The Battle of Rendili was a battle of the Clone Wars, occurring thirty months after the First Battle of Geonosis, in 20 BBY.
- The First Battle of Kamino, also known as the Defense of Kamino, the Assault on Kamino, or simply as the Battle of Kamino, took place two months after the First Battle of Geonosis.
- The Conquest of Raxus Prime was a battle initiated by the Galactic Republic during the Clone Wars, shortly after the First Battle of Geonosis.
- Ronto was a clone trooper copilot who fought in the First Battle of Geonosis during the Clone Wars.
Question: What is the name of the settlement where Anakin Skywalker and his mother lived?
Retrieved documents:


- The Skywalker home was where the slaves Anakin Skywalker and his mother Shmi Skywalker Lars lived in Mos Espa on the desert planet Tatooine.
- In 32 BBY, she helped Anakin Skywalker return to the Jedi Temple after he helped her son repair their malfunctioning nanny droid.
- During the Invasion of Naboo, Mos Espa was also home to a number of slaves, including Anakin Skywalker and his mother, Shmi.
- Her mother was killed during the Battle of Lanteeb when Jedi Master Obi-Wan Kenobi and Jedi Knight Anakin Skywalker were on the planet to try and stop Separatist General Lok Durd.
- Finn's mother lived with him on Coruscant.
Question: Which planet did Darth Sidious represent as senator?
Retrieved documents:


- The Sith Lord Darth Sidious dispatched his Sith apprentice, Darth Maul, to kill Cion and his associates after the Devaronian threatened to blackmail Senator Sheev Palpatine&mdash;Sidious's public persona&mdash;with information on his underworld dealings.
- Senator Pax Teem of the Gran Protectorate was murdered in the embassy by Darth Plagueis' apprentice, Darth Sidious, thus removing an opponent to the Sith Lords.
- Cion discovered underworld dealings of Senator Sheev Palpatine&mdash;in reality, the Sith Lord Darth Sidious, who sent his Sith apprentice, Darth Maul, to kill the Devaronian.
- He also mentioned that his Sith Master, Darth Plagueis, had a minor role in these events.
- The volume detailed how Sidious, under his public persona of the Naboo Senator Palpatine, managed to manipulate several important figures and events to ensure his rise to power.


Conclusion



I combined BM25 with SentenceTransformer to find relevant sentences.



Most questions returned useful information, but some results were less relevant.

For the Luke Skywalker question, the retrieved sentences mainly describe his belongings rather than his identity.

This shows that hybrid retrieval works, but the accuracy can still be improved.


## Putting it all together

Now that we have a functioning retriever model, we can connect it to the generative language model employed to produce the responses.

**Task 4** (4 points): Implement the `RetrievalAugmentedResponseGenerator`. Given an initial question, the method should first retrieve relevant sentences using the `HybridRetriever` we have just developed. Then, it should expand the initial prompt using the provided template (you are of course free to edit or adapt it as you see fit). This expanded prompt should then be tokenized and fed as input to the LLM in the same way as before.

In [10]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

PROMPT_TEMPLATE = "You are given the following facts about Star Wars:\n-{retrieved_sentences}\n\nWith the help of those facts and your knowledge of Star Wars, answer the following question in 1 or 2 sentences: '{query}'\nAnswer:"


class RetrievalAugmentedResponseGenerator:

    def __init__(
        self,
        model_name="google/gemma-4-E2B-it",
        doc_filename="starwars.json",
        encoder_model="all-MiniLM-L6-v2",
        cache_dir="/fp/projects01/ec403/hf_models",
    ):
        """Loads the tokenizer, pretrained causal LM for the given model, along with the
        hybrid sparse-dense retriever model populated with the documents in doc_filename.
        """

        #Load the response generator from Task 1


        self.generator = BasicResponseGenerator(model_name= model_name, cache_dir=cache_dir)


        #Load the hybird retriever from Task 3

        self.retriever = HybridRetriever(filename=doc_filename, encoder_model=encoder_model, cache_dir=os.path.expanduser("~/hf_models"))

        
    

    def get_response(self, query: str, max_length: int = 50, k=5) -> str:
        """Given a prompt, retrieve k relevant sentences, generate a response (of a maximum
        max_length tokens) and return it.
        Only the response should be returned, not the text of the prompt itself
        """

        #Retrieve k relevant sentences from Task 3
        retrieved_sentences = self.retriever.search(query, k=k)

        #Combine retrieved sentences 
        context = "\n-".join(retrieved_sentences)
        
        #Construct the prompt
        prompt = PROMPT_TEMPLATE.format(retrieved_sentences=context, query=query)
        
        # Generate the answer
        response = self.generator.get_response(prompt, max_length=max_length)
        return response
        


agent = RetrievalAugmentedResponseGenerator()

#Test

print(agent.get_response("Where is Niima Outpost?"))
print(agent.get_response("Who is Luke Skywalker?"))



Loading weights: 100%|██████████| 1951/1951 [00:02<00:00, 747.19it/s]


<class 'dict'>
{'Brianna': 'Brianna, also known as "The Last Handmaiden," was a half-Human, half-Echani hybrid fighter of Jedi heritage. The illegitimate daughter to the famous Echani General Yusanis and Jedi Master Arren Kae, both of whom fought in the Mandalorian Wars, she later served along with her five half-sisters as the Handmaiden Sisters under Jedi Master Atris in the period following the Jedi Civil War. During her service to Atris at the Jedi Academy on Telos IV, Brianna met Meetra Surik, also known a


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7283.86it/s]            


Niima Outpost was a junkyard settlement located on the desert planet of Jakku.


Based on the provided facts, Luke Skywalker is a figure whose identity is complicated by secret machinations, as Blackhole was secretly planning to become him and created events to establish Luke as the legitimate heir to the Empire. He is also associated with specific personal


Conclusion
I combined the HybridRetriever with the response generator to answer questions.
The model answered the Niima Outpost question correctly, but the Luke Skywalker answer was less relevant.
This shows that the answer quality depends on the retrieved information.

The last step is to test our system end-to-end:

In [11]:
for question in questions:
    print("Question:", question)
    print("Answer:", agent.get_response(question))
    print("-------")



Question: Who is Luke Skywalker?


Answer: Based on the provided facts, Luke Skywalker is a character who is secretly being groomed by Blackhole to become the next legitimate heir to the Empire, and he is associated with a blue gundark-skin satchel and his own lightsabers and
-------
Question: Where is the Niima Outpost in Star Wars?


Answer: The Niima Outpost is a junkyard settlement and the only landing field on the desert planet of Jakku in the Western Reaches of the galaxy.
-------
Question: Have you heard of Nute Gunray? Who is he?


Answer: Yes, I have heard of Nute Gunray; he was a prominent figure in the Star Wars universe, specifically as the Viceroy of the Trade Federation.
-------
Question: What kind of planet is Kashyyyk, and who discovered it?


Answer: Kashyyyk is a wroshyr tree-covered forest planet and the homeworld of the Wookiee species. It was discovered by the Czerka Corporation, who later enslaved the Wookiee population on the planet.
-------
Question: Who are Condlurans, and can you give 2-3 names of known Condlurans?


Answer: Condlurans are a sentient species that exists within the galaxy. While the provided facts mention specific Condlurans like the Condluran Briff and Freck, there are no other specific names of known Condlurans given.
-------
Question: What can you tell me about the First Battle of Geonosis?


Answer: The First Battle of Geonosis was the initial major conflict of the Clone Wars, taking place in 22 BBY between the Confederacy of Independent Systems and the Galactic Republic on the planet Geonosis. This battle is also significant because it marked
-------
Question: What is the name of the settlement where Anakin Skywalker and his mother lived?


Answer: Anakin Skywalker and his mother, Shmi Skywalker Lars, lived in Mos Espa on the desert planet Tatooine.
-------
Question: Which planet did Darth Sidious represent as senator?


Answer: Darth Sidious represented the planet Naboo as Senator Palpatine.
-------


Conclusion

I combined the HybridRetriever with the response generator to build a RAG system.

Most questions were answered correctly, but the Luke Skywalker answer contained incorrect information.

This shows that RAG can improve answers, but its performance depends on the retrieved information and how the model uses it.

**Task 5** (4 points): If you have implemented your model correctly, the system should answer correctly to at least a few questions. But it is still far from perfect, and some of the answers are flat-out wrong. Suggest 2-3 ways one could improve the current system and get even better answers. You don't need to implement anything, simply flesh out a few ideas you believe are worth trying out.

_(of course, it is even better if you actually try to implement those ideas and evaluate their influence on the quality of the system responses!)_

Answer 

Conclusion



Change the number of retrieved sentences
In Task 4, we have k=5. We can try k=3, k=10 or other values.
If we use a larger k, the model will get more context and information. 
However, more sentences may also contain irrelevant information, which could make the answers worse.

Improve the retrieval method
We use BM25 and dense retrieval in Task 3.
We could try using a different SentenceTransformer model to find more relevant sentences.
This may help the model get better information.


Improve the prompt
We could change the prompt to tell the model to focus more on the retrieved information.
If the model does not have enough information, 
it should say that it does not know the answer instead of generating incorrect information.